# Learn representations with I-JEPA

You will train an I-JEPA encoder: a ViT that sees the visible patches of an image and predicts, in representation space, the embeddings of the patches that were hidden. No pixel reconstruction and no augmentation pipeline. After training you measure the representation with a linear probe and a kNN probe against a shuffled-label control, read the collapse telemetry, and pull the nearest neighbours of query images out of the frozen embeddings.

A context encoder sees only some patches, a target encoder sees the whole image, and a narrow predictor maps the context embeddings plus the target positions to the target embeddings. The target encoder is the EMA of the context encoder rather than a trained module, so the targets keep improving and there is no pixel-level shortcut.

Two routes share every cell but the configuration:

- **Offline route (default, runs here).** Generated stripe images in four classes (orientation and stripe width), 16 pixels, a small encoder, a few hundred steps on a CPU.
- **Oxford Flowers route.** `ROUTE = "flowers"` reads the prepared dataset from notebook 02 at 224 pixels with a ViT-S width encoder for 100 epochs, which wants a GPU or TPU for under an hour. That route was not executed while writing this notebook.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew" {jax_spec}

In [ ]:
ROUTE = "offline"          # "offline" runs anywhere; "flowers" needs the prepared dataset and an accelerator
RUN_DIR = "runs/06-jepa"

if ROUTE == "offline":
    IMAGE_SIZE = 16
    PATCH_SIZE = 4
    BATCH_SIZE = 32
    STEPS = 400
    LEARNING_RATE = 1e-3
    EMB_FEATURES = 32
    NUM_LAYERS = 2
    NUM_HEADS = 2
    NUM_TARGET_BLOCKS = 1
    BLOCK_SCALE = (0.25, 0.25)  # one 2x2 block of the 4x4 grid
    CLASSES = 4
    DTYPE, ATTENTION = "float32", "xla"
else:
    FLOWERS_PATH = "~/dew-data/tfds-arrayrecord/oxford_flowers102/2.1.1"  # builder.data_dir from preparation
    IMAGE_SIZE = 224
    PATCH_SIZE = 16
    BATCH_SIZE = 64
    STEPS = 12_400             # 100 epochs of 124 steps
    LEARNING_RATE = 1e-3
    EMB_FEATURES = 384
    NUM_LAYERS = 12
    NUM_HEADS = 6
    NUM_TARGET_BLOCKS = 4
    BLOCK_SCALE = (0.15, 0.2)
    CLASSES = 102
    DTYPE, ATTENTION = "bfloat16", "auto"
GRID = (IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE)
SEED = 0

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The data

Batches carry uint8 images under `image` and an integer class under `label`, which the probes score against. In I-JEPA the variation comes from the masking, so no flips or colour jitter are applied.

The offline route generates stripe images whose class is the orientation and the stripe width. The flowers route reads prepared ArrayRecords (the preparation command is in notebook 02) and carries the class index through as `label`. Validation records are held out of the head of the dataset, so the probes never score an image the encoder trained on.

In [ ]:
import numpy as np
from dew.data import Dataset, Loading


def stripe_images(count, size, seed):
    """uint8 [count, size, size, 3] stripes; class = 2 * orientation + (wide stripes)."""
    rng = np.random.default_rng(seed)
    images = np.zeros((count, size, size, 3), np.uint8)
    labels = rng.integers(0, 4, count)
    for i in range(count):
        colour = rng.integers(64, 256, 3)
        period = 2 if labels[i] % 2 == 0 else 4
        offset = int(rng.integers(0, period * 2))
        bands = ((np.arange(size) + offset) // period) % 2 == 0
        pattern = bands[:, None] if labels[i] < 2 else bands[None, :]
        images[i][np.broadcast_to(pattern, (size, size))] = colour
    return images, labels.astype(np.int32)


if ROUTE == "offline":
    train_images, train_labels = stripe_images(1024, IMAGE_SIZE, seed=SEED)
    val_images, val_labels = stripe_images(256, IMAGE_SIZE, seed=SEED + 1)

    class StripeBatches:
        def __init__(self):
            self.index = 0

        def __iter__(self):
            return self

        def __next__(self):
            rows = np.random.default_rng(SEED + self.index).choice(
                len(train_images), BATCH_SIZE, replace=False)
            self.index += 1
            return {"image": train_images[rows], "label": train_labels[rows]}

        def get_state(self):
            return str(self.index).encode()

        def set_state(self, state):
            self.index = int(state.decode())

    def val_batches():
        for start in range(0, len(val_images), BATCH_SIZE):
            yield {"image": val_images[start:start + BATCH_SIZE],
                   "label": val_labels[start:start + BATCH_SIZE]}

    data = Dataset(train=StripeBatches, val=val_batches,
                   records=len(train_images), batch=BATCH_SIZE)
else:
    from dew.data import OxfordFlowers
    data = OxfordFlowers(path=FLOWERS_PATH, image_size=IMAGE_SIZE, augmentation="none",
                         val_batches=4,
                         loading=Loading(workers=4, threads=4, read_buffer=16, worker_buffer=2)
                         ).load(batch=BATCH_SIZE)

batch = next(iter(data.train()))
print("train records:", data.records, "|", batch["image"].shape, batch["image"].dtype,
      "| labels:", batch["label"][:8])

## The mask

Each image gets `NUM_TARGET_BLOCKS` rectangular blocks of patches to predict, drawn with a random scale and aspect ratio, and the context is a random subset of everything left. The geometry is resolved once for the patch grid and each step samples block positions from it, so every mask has the same shape and the training step stays compiled.

The cell prints one sampled mask over the grid: `.` is context the encoder sees, `#` is a target block it predicts, and `-` is dropped.

In [ ]:
from dew.objectives.jepa import multi_block_mask

mask = multi_block_mask(GRID, num_targets=NUM_TARGET_BLOCKS, scale=BLOCK_SCALE)
print("grid", GRID, "| context tokens:", mask.num_context,
      "| targets:", mask.num_targets, "blocks x", mask.block_area, "tokens each")

context_idx, target_idx = mask.sample(jax.random.key(SEED), 1)
view = np.full(GRID[0] * GRID[1], "-")
view[np.asarray(context_idx[0])] = "."
view[np.asarray(target_idx).reshape(-1)] = "#"
print("\n".join("".join(row) for row in view.reshape(GRID)))

## Encoder, predictor, objective

The encoder is a ViT (`jepa_encoder`). The predictor is a narrower transformer (`jepa_predictor`) that reads the context embeddings plus mask tokens standing in for the targets and outputs embeddings at the encoder's width. `JepaObjective` ties them together with the mask and the sample field; its `EMASpec` restricts the trainer's EMA to the `context_encoder` subtree, and that EMA copy is the target encoder.

The loss is the mean squared distance between predictions and layer-normalized targets, in fp32. The objective also reports two collapse numbers on every step: `repr_std`, the spread of the embeddings across a batch, which goes to zero when the encoder stops distinguishing inputs, and `repr_cov_offdiag`, which rises when the embedding dimensions become redundant.

In [ ]:
from dew import Field, models
from dew.objectives.jepa import JepaObjective

encoder = models.build("jepa_encoder", patch_size=PATCH_SIZE, emb_features=EMB_FEATURES,
                       num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
                       dtype=DTYPE, attention_impl=ATTENTION)
predictor = models.build("jepa_predictor", grid=GRID, emb_features=EMB_FEATURES,
                         predictor_features=EMB_FEATURES // 2,
                         num_layers=max(1, NUM_LAYERS // 2), num_heads=NUM_HEADS,
                         dtype=DTYPE, attention_impl=ATTENTION)
objective = JepaObjective(encoder, predictor, mask=mask,
                          sample=Field("image", (IMAGE_SIZE, IMAGE_SIZE, 3)),
                          momentum_steps=STEPS)

variables = jax.eval_shape(objective.init, jax.random.key(0))
for name, tree in variables["params"].items():
    print(f"{name}: {sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(tree)) / 1e6:.2f}M parameters")

## Training with the probes as metrics

`Trainer` is the one the other notebooks use. At every `eval_every` the objective's evaluation embeds the held-out images with the frozen EMA encoder and returns `Representations` (pooled features plus labels); each probe metric fits on the first half of a validation batch and scores the second half, then averages over the pass. Chance on the offline classes is 25 percent, on the flowers 1 percent.

The telemetry cell after training reads the same evaluation, so the collapse numbers are measured on held-out images.

In [ ]:
import optax
from dew import Checkpoints, Trainer, metrics

trainer = Trainer(objective, optax.adamw(LEARNING_RATE), key=jax.random.key(SEED),
                  checkpoints=Checkpoints(RUN_DIR))
state = trainer.fit(data, steps=STEPS, log_every=max(1, STEPS // 5),
                    eval_every=max(1, STEPS // 2), checkpoint_every=STEPS,
                    metrics=(metrics.linear_probe(CLASSES), metrics.knn_probe(CLASSES)))
print("optimizer updates:", int(state.updates))

## Telemetry and probes on held-out images

`objective.evaluate` returns the pooled EMA-encoder embeddings of a batch. `representation_health` on them is the collapse telemetry: `repr_std` should stay well away from zero and `repr_cov_offdiag` should stay low. The probe functions score the same embeddings; the shuffled-label control beside each number is what makes it readable, since a probe with more dimensions than samples can fit anything it is shown, and the honest question is how far the real-label score sits above permuted labels.

In [ ]:
from dew.objectives.base import Step
from dew.objectives.jepa import representation_health
from dew.objectives.jepa.probes import knn_probe_accuracy, linear_probe_accuracy

features, labels = [], []
for batch in data.val():
    scored = objective.evaluate(state.params, batch,
                                Step(step=state.step, key=jax.random.key(1), ema=state.averaged))
    features.append(np.asarray(scored.features))
    labels.append(np.asarray(scored.labels))
features, labels = np.concatenate(features), np.concatenate(labels)
print("held-out embeddings:", features.shape)

health = representation_health(features)
print(f"repr_std {float(health['repr_std']):.3f} | repr_cov_offdiag {float(health['repr_cov_offdiag']):.4f}")

shuffled = np.random.default_rng(0).permutation(labels)
for name, probe in (("linear probe", linear_probe_accuracy), ("kNN probe", knn_probe_accuracy)):
    real = float(probe(features, labels, CLASSES))
    control = float(probe(features, shuffled, CLASSES))
    print(f"{name}: {real:.3f} on the real labels, {control:.3f} on shuffled labels")

## Nearest neighbours

The point of the encoder is the embedding space, so look at it directly: normalise the embeddings and take cosine neighbours of a few query images. Each row is one query followed by its four nearest neighbours, with their labels. Images of the same class clustering together is what the probes measured numerically.

In [ ]:
from PIL import Image

try:
    from IPython.display import display
except ModuleNotFoundError:  # running the cells as a plain script
    def display(image):
        print(f"image {image.width}x{image.height}")

held_out = np.concatenate([batch["image"] for batch in data.val()])
normalised = features / (np.linalg.norm(features, axis=-1, keepdims=True) + 1e-8)
rows, agreement = [], []
for query in range(4):
    neighbours = np.argsort(-(normalised @ normalised[query]))[1:5]
    rows.append(np.concatenate([held_out[query], *held_out[neighbours]], axis=1))
    agreement.append(float(np.mean(labels[neighbours] == labels[query])))
    print(f"query label {labels[query]} | neighbour labels {labels[neighbours].tolist()}")
grid = Image.fromarray(np.concatenate(rows, axis=0))
display(grid.resize((grid.width * 4, grid.height * 4), Image.NEAREST))
print("fraction of neighbours sharing the query's label:", np.mean(agreement))

## Keeping the encoder

The thing to keep from a JEPA run is the EMA of the context encoder, without the predictor. `save_params` writes it as a safetensors file under the names the tree uses; `load_params` reads it back. The trainer's checkpoints under `RUN_DIR` hold the full train state, including the optimizer, if you want to resume.

In [ ]:
from pathlib import Path

from dew.interop import load_params, save_params

Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
encoder_params = state.averaged["params"]["context_encoder"]
save_params(encoder_params, f"{RUN_DIR}/encoder.safetensors")
reloaded = load_params(f"{RUN_DIR}/encoder.safetensors")
again = objective.encode(reloaded, (held_out[:8].astype(np.float32) - 127.5) / 127.5)
print("wrote", f"{RUN_DIR}/encoder.safetensors", "| reloaded features:", np.asarray(again).shape)

## Where to go next

The paper's ViT-H/16 trains 300 epochs on ImageNet with more target blocks; the knobs are the ones this notebook set once (`NUM_TARGET_BLOCKS`, `BLOCK_SCALE`, the encoder width, `STEPS`). `recipes/jepa/train.py` runs the same configuration from the command line, and `jepa_video_encoder` with a factorized predictor does the same job on video clips.